In [1]:
from pyspark.sql import functions as F

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 3, Finished, Available, Finished, False)

In [2]:
bronze_path = 'Files/bronze/consumer_complaints/Consumer_Complaints.csv'

df_bronze = (
    spark.read
    .option('header','true')
    .option('inferschema','true')
    .option('multiline','true')
    .option('quote','"')
    .option('escape','"')
    .option('unescapedQuoteHandling', 'STOP_AT_CLOSING_QUOTE')
    .option('nullValue','N/A')
    .csv(bronze_path)
)

display(df_bronze)

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dbbd6149-95fd-45a8-ba38-17cbcae0a7d2)

In [3]:
print('Row COunt',df_bronze.count())
print('Column COunt',len(df_bronze.columns))

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 5, Finished, Available, Finished, False)

Row COunt 903983
Column COunt 18


In [4]:
df_bronze.printSchema()

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 6, Finished, Available, Finished, False)

root
 |-- Date received: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Sub-product: string (nullable = true)
 |-- Issue: string (nullable = true)
 |-- Sub-issue: string (nullable = true)
 |-- Consumer complaint narrative: string (nullable = true)
 |-- Company public response: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP code: string (nullable = true)
 |-- Tags: string (nullable = true)
 |-- Consumer consent provided?: string (nullable = true)
 |-- Submitted via: string (nullable = true)
 |-- Date sent to company: string (nullable = true)
 |-- Company response to consumer: string (nullable = true)
 |-- Timely response?: string (nullable = true)
 |-- Consumer disputed?: string (nullable = true)
 |-- Complaint ID: integer (nullable = true)



**1. Complaint ID**

In [4]:
complaint_id_check = df_bronze.agg(
                            F.count('*').alias('Total Rows'),
                            F.count('Complaint ID').alias('Non_Null_Complaint_IDs'),
                            F.countDistinct('Complaint ID').alias('Distinct_Complaint_IDs'),
                            F.sum(
                                F.when(F.col('Complaint ID').isNull(),1).otherwise(0)
                            ).alias('Null_Complaint_IDs')
                        )

display(complaint_id_check)

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c1406d8-7f79-4a29-a08d-0cd72bb6635d)

In [5]:
total_rows = df_bronze.count()
null_summary = []
for column_name in df_bronze.columns:
    null_count = df_bronze.filter(F.col(column_name).isNull()).count()
    total_count = df_bronze.select(column_name).count()

    null_summary.append(
        (
            column_name,
            null_count,
            total_count,
            round((null_count/total_rows)*100,2)
        )
    )
    null_df = spark.createDataFrame(
    null_summary,
    ["Column_Name", "Null_Count", "Total_Count", "Null_Percentage"]
    ) 

display(null_df)

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 23b2928b-907c-4b07-9204-19fc60b6b3b3)

In [6]:
df_silver = df_bronze

df_silver = df_silver.withColumnRenamed(
    "Complaint ID",
    "complaint_id"
)
display(df_silver)

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ff99012-28e0-4672-a0cb-519a4cf7789a)

**2. DATE RECEIVED**

In [7]:
date_received_check = df_bronze.select(
                        F.count("Date received").alias("Total_Rows"),
                        F.sum(F.when(F.col("Date received").isNull(),1).otherwise(0)).alias("Null_Count"),
                        F.min("Date received").alias("Min_Value"), 
                        F.max("Date received").alias("Max_Value")
                    )
display(date_received_check)

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fb78c0a6-f946-4d71-aa2b-1dc8a6b8abb8)

In [9]:
df_silver = df_silver.withColumn(
             "date_received",
             F.to_date(F.col("Date received"),"M/d/yyyy")
)

display(df_silver.select(
            "Date received",
            "date_received"
    )
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0eb443d4-addf-4dd1-ab78-6679e2b26b5f)

In [8]:
df_silver = df_silver.drop("Date received")
display(df_silver)

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c909892-1e4b-4e4f-b8fa-35293cd6a872)

**3. Date sent to company**

In [9]:
display(df_silver.select('Date sent to company'))

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f7d25090-1aad-484c-bab6-f6fd4e554262)

In [10]:
date_sent_check = df_bronze.select(
                        F.count("Date sent to company").alias("Total_Rows"),
                        F.sum(F.when(F.col("Date received").isNull(),1).otherwise(0)).alias("Null_Count"),
                        F.min("Date sent to company").alias("Min_Value"), 
                        F.max("Date sent to company").alias("Max_Value")
                    )
display(date_sent_check)

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fce0a336-0c64-4a40-932d-4540dcbd0c33)

In [11]:
df_silver = df_silver.withColumn(
             "date_sent_to_company",
             F.to_date(F.col("Date sent to company"),"M/d/yyyy")
)

display(df_silver.select(
            "Date sent to company",
            "date_sent_to_company"
    )
)

StatementMeta(, a2f4f19b-6b78-4d47-aeff-8bbde6d06c7d, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f21ee297-c975-41c5-8114-b5e3b7c7f25c)

In [14]:
df_silver = df_silver.drop("Date sent to company")
display(df_silver)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 231cc81b-5903-417b-9863-fc480ffe3644)

In [15]:
date_sequence_check = (
    df_silver.filter(F.col('date_sent_to_company') < F.col('date_received'))
)

display(date_sequence_check.count())

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 17, Finished, Available, Finished, False)

7052

In [16]:
df_quarantine = (
    df_silver.filter(
        F.col('date_sent_to_company') < F.col('date_received'))
    .withColumn('dq_status',F.lit("FAIL"))
    .withColumn('dq_rule',F.lit('DATE SEQUENCE'))
    .withColumn('dq_reason',F.lit('Date sent to company is earlier than date received')
)
)
display(df_quarantine)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3e89f187-7dea-4e33-a27f-7fdb8a5eb407)

In [17]:
df_silver_valid = (
    df_silver.filter(F.col('date_sent_to_company') >= F.col('date_received'))
)

display(df_silver_valid.count())
display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 19, Finished, Available, Finished, False)

896931

SynapseWidget(Synapse.DataFrame, 5414ef6e-37e1-46d6-93e0-c90422b431cc)

**4. PRODUCT**

In [ ]:
df_silver_valid = df_silver_valid.withColumn(
     "product_standardized",
    
  
    F.when(
        ((F.col("product") == "Prepaid card") &
         F.trim(F.col("Sub-product")).isin("Transit card", "Other special purpose card")) |
        ((F.col("product") == "Bank account or service") &
         (F.trim(F.col("Sub-product")) == "Cashing a check without an account")),
        F.col("product")
    )

     .when(
        (F.col("product") == "Prepaid card") &
        (F.trim(F.col("Sub-product")) == "Mobile wallet"),
        "Money transfer, virtual currency, or money service"
    )

    .when(
        (F.col("product") == "Consumer Loan" ) &
        F.trim(F.col("Sub-product")).isin("Vehicle loan", "Vehicle lease"),
        "Vehicle loan or lease"
    )
    .when(
        (F.col("product") == "Consumer Loan") &
        F.trim(F.col("Sub-product")).isin(
            "Installment loan", "Pawn loan", "Personal line of credit", "Title loan"),
        "Payday loan, title loan, or personal loan"
    )
    
    .when(
        (F.col("product") == "Other financial service") &
        (F.trim(F.col("Sub-product")) == "Credit repair"),
        "Credit reporting, credit repair services, or other personal consumer reports"
    )

    .when(
        (F.col("product") == "Other financial service") &
        F.trim(F.col("Sub-product")).isin(
            "Check cashing", "Debt settlement", "Foreign currency exchange",
            "Money order", "Refund anticipation check", "Traveler’s / cashier’s checks",
            "Traveler's/Cashier's checks", "Traveler's/Cashier's checks"),
        "Money transfer, virtual currency, or money service"
    )
    
    .when(F.col("product").isin("Credit card", "Prepaid card"),
          "Credit card or prepaid card")
    .when(F.col("product") == "Bank account or service", "Checking or savings account")
    .when(F.col("product") == "Credit reporting",
          "Credit reporting, credit repair services, or other personal consumer reports")
    .when(F.col("product") == "Payday loan", "Payday loan, title loan, or personal loan")
    .when(F.col("product") == "Money transfers",
          "Money transfer, virtual currency, or money service")

    .otherwise(F.col("product"))

)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 55d07abd-ef3a-4332-93af-d473b3ac69d6)

In [19]:
df_silver_valid = df_silver_valid.withColumn('product',F.trim('Product'))

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 93e66cc3-34a5-44e2-925f-df3aaafec79d)

**5.SUB_PRODUCT**

In [20]:
df_silver_valid = df_silver_valid.withColumnRenamed(
    "Sub-product",
    "sub_product"
)

df_silver_valid = df_silver_valid.withColumn(
     "sub_product",
     F.trim(F.col("sub_product"))
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5b5979d3-7f5b-415f-b8c6-a6a2cad5d76f)

In [21]:
display(df_silver_valid.groupBy("sub_product").count().orderBy(F.col('sub_product').desc()))

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d54c0fb-e842-4808-a881-cb2a6990229a)

In [22]:
df_silver_valid = df_silver_valid.withColumn(
     "sub_product_standardized",

F.when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Auto'),
        'Auto debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Credit card'),
        'Credit card debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Federal student loan'),
        'Federal student loan debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Medical'),
        'Medical debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Mortgage'),
        'Mortgage debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Non-federal student loan'),
        'Private student loan debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Other', 'Other (i.e. phone, health club, etc.)'),
        'Other debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Payday loan'),
        'Payday loan debt'
    )

    .when(
        (F.col("product") == 'Mortgage') &
        F.col("sub_product").isin('Conventional adjustable mortgage (ARM)', 'Conventional fixed mortgage'),
        'Conventional home mortgage'
    )
    .when(
        (F.col("product") == 'Mortgage') &
        F.col("sub_product").isin('Home equity loan or line of credit'),
        'Home equity loan or line of credit (HELOC)'
    )
    .when(
        (F.col("product") == 'Mortgage') &
        F.col("sub_product").isin('Other', 'Other mortgage'),
        'Other type of mortgage'
    )
    
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('General purpose card'),
        'General-purpose prepaid card'
    )
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('Gift or merchant card'),
        'Gift card'
    )
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('Electronic Benefit Transfer / EBT card', 'Government benefit payment card'),
        'Government benefit card'
    )
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('ID prepaid card'),
        'Student prepaid card'
    )
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('Mobile wallet'),
        'Mobile or digital wallet'
    )

    .when(
        (F.col("product") == 'Bank account or service') &
        F.col("sub_product").isin('(CD) Certificate of deposit'),
        'CD (Certificate of Deposit)'
    )
    .when(
        (F.col("product") == 'Bank account or service') &
        F.col("sub_product").isin('Other bank product / service', 'Other bank product/service'),
        'Other banking product or service'
    )

    .when(
        (F.col("product") == 'Consumer loan') &
        F.col("sub_product").isin('Vehicle loan'),
        'Loan'
    )
    .when(
        (F.col("product") == 'Consumer loan') &
        F.col("sub_product").isin('Vehicle lease'),
        'Lease'
    )

    .when(
        (F.col("product") == 'Student loan') &
        F.col("sub_product").isin('Non-federal student loan'),
        'Private student loan'
    )
    
    .when(
        (F.col("product") == "Other financial service") &
        (F.col("sub_product") == "Credit repair"),
        "Credit repair services"
    )
    .when(
        (F.col("product") == "Other financial service") &
        (F.col("sub_product") == "Check cashing"),
        "Check cashing service"
    )
    .when(
        (F.col("product") == "Other financial service") &
        F.regexp_replace(F.col("sub_product"), "\u2019", "'").isin(
            "Traveler's / cashier's checks",
            "Traveler's/Cashier's checks"
        ),
        "Traveler's check or cashier's check"
    )

    .when(
        (F.col("product") == 'Consumer Loan') &
        F.col("sub_product").isin('Vehicle loan'),
        'Loan'
    )
    .when(
        (F.col("product") == 'Consumer Loan') &
        F.col("sub_product").isin('Vehicle lease'),
        'Lease'
    )

    .otherwise(F.col("sub_product"))
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a664bd96-f50b-480b-b5e3-83cd8448fdd6)

In [23]:
display(df_silver_valid.groupBy(
    "product", "sub_product", "product_standardized", "sub_product_standardized"
).count().orderBy("product", "sub_product"))

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2da8c907-d50f-4845-b5e2-fefd67fa5718)

**ISSUE**

In [24]:
display(df_silver_valid.select('Issue').distinct().sort('Issue'))

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eb28c3c9-8eca-4b3f-a726-7f25d2276b87)

In [25]:
display(df_silver_valid.groupBy("Issue").count().orderBy(F.col("Issue").desc()))

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 332d3511-42f3-488e-8225-19f6f98afe37)

In [26]:
df_silver_valid = df_silver_valid.withColumn(
    "issue_standardized",

   F.when(
        F.col("issue") == "Customer service / Customer relations",
        "Customer service/Customer relations"
     ) 
     .when(
        F.col("issue") == "Overdraft, savings or rewards features",
        "Overdraft, savings, or rewards features"
     )

     .when(
        (F.col("issue") == "Problem with an overdraft") & 
        (F.col("product") == "Credit card or prepaid card"),
        "Problem with overdraft"
     )
     .when(
        (F.col("issue") == "Charged fees or interest I didn't expect") &
        (F.col("product") == "Payday loan"),
        "Charged fees or interest you didn't expect"
     )
     .when(
        (F.col("issue") == "Received a loan I didn't apply for") & 
        (F.col("product") == "Payday loan"),
        "Received a loan you didn't apply for"
     )

     .when(
        (F.col("issue") == "Dealing with my lender or servicer") &
        (F.col("product") == "Student loan"),
        "Dealing with your lender or servicer"
     )

     .when(
        (F.col("issue") == "Incorrect information on credit report") &
        (F.col("product") == "Credit reporting"),
        "Incorrect information on your report"
     )

     .when(
        (F.col("issue") == "Improper use of my credit report") &
        (F.col("product") == "Credit reporting"),
        "Improper use of your report"
     )
     .when(
        (F.col("issue") == "Unable to get credit report/credit score") &
        (F.col("product") == "Credit reporting"),
        "Unable to get your credit report or credit score"
    )

    .when(
        (F.col("issue") == "Credit reporting company's investigation") &
        (F.col("product") == "Credit reporting"),
        "Problem with a credit reporting company's investigation into an existing problem"
    )

    .when(
        (F.col("issue") == "Problems caused by my funds being low") &
        (F.col("product") == "Bank account or service"),
        "Problem caused by your funds being low"
    )

    .when(
        (F.col("issue") == "Cont'd attempts collect debt not owed") &
        (F.col("product") == "Debt collection"),
        "Attempts to collect debt not owed"
    )
    .when(
        (F.col("issue") == "Other service issues") &
        (F.col("product").isin(
            "Money transfers",
            "Virtual currency"
        )),
        "Other service problem"
    )

    .when(
        (F.col("issue") == "Other transaction issues") &
        (F.col("product").isin(
            "Money transfers",
            "Virtual currency"
        )),
        "Other transaction problem"
    )
    
    .otherwise(F.col("issue"))   
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 53b5d1c6-56bd-4507-b1d2-4e90919ab543)

**SUB_ISSUE**

In [27]:
sub_issue_profile = (
    df_silver_valid.groupBy("Sub-issue")
                   .count()
                   .orderBy(F.col("Sub-issue").desc())
)
display(sub_issue_profile)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4938b957-d400-4f2e-b0d4-7a06d75fe355)

In [28]:
df_silver_valid = df_silver_valid.withColumnRenamed("Sub-issue","sub_issue").withColumn("sub_issue",F.trim(F.col("sub_issue")))
display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fe1981eb-a93e-42fd-8ba2-c201a472b1a8)

In [29]:
df_silver_valid = df_silver_valid.withColumn(
    "sub_issue_standardized",

    F.when(
            (F.col("sub_issue") == "Account status") &
            (F.col("product") == "Credit reporting"),
            "Account status incorrect"
        )

    .when(
        (F.col("sub_issue") == "Information is not mine") &
        (F.col("product") == "Credit reporting"),
        "Information belongs to someone else"
    )
    .when(
        (F.col("sub_issue") == "Personal information") &
        (F.col("product") == "Credit reporting"),
        "Personal information incorrect"
    )
    .when(
        (F.col("sub_issue") == "Public record") &
        (F.col("product") == "Credit reporting"),
        "Public record information inaccurate"
    )

    .when(
        (F.col("sub_issue") == "Problem getting my free annual report") &
        (F.col("product") == "Credit reporting"),
        "Problem getting your free annual credit report"
    )

    .when(
        (F.col("sub_issue") == "No notice of investigation status/result") &
        (F.col("product") == "Credit reporting"),
        "Was not notified of investigation status or results"
    )

    .when(
        (F.col("sub_issue") == "Report shared with employer w/o consent") &
        (F.col("product") == "Credit reporting"),
        "Report provided to employer without your written authorization"
    )

    .when(
        (F.col("sub_issue") == "Received marketing offer after opted out") &
        (F.col("product") == "Credit reporting"),
        "Received unsolicited financial product or insurance offers after opting out"
    )

    .when(
        (F.col("sub_issue") == "Receiving unwanted marketing/advertising") &
        (F.col("product") == "Credit reporting"),
        "Received unwanted marketing or advertising"
    )

    .when(
        (F.col("sub_issue") == "Debt is not mine") &
        (F.col("product") == "Debt collection"),
        "Debt is not yours"
    )

    .when(
        (F.col("sub_issue") == "Debt resulted from identity theft") &
        (F.col("product") == "Debt collection"),
        "Debt was result of identity theft"
    )

    .when(
        (F.col("sub_issue") == "Debt was discharged in bankruptcy") &
        (F.col("product") == "Debt collection"),
        "Debt was already discharged in bankruptcy and is no longer owed"
    )

    .when(
        (F.col("sub_issue") == "Attempted to/Collected exempt funds") &
        (F.col("product") == "Debt collection"),
        "Collected or attempted to collect exempt funds"
    )

     .when(
        (F.col("sub_issue") == "Not given enough info to verify debt") &
        (F.col("product") == "Debt collection"),
        "Didn't receive enough information to verify debt"
    )

    .when(
        (F.col("sub_issue") == "Right to dispute notice not received") &
        (F.col("product") == "Debt collection"),
        "Didn't receive notice of right to dispute"
    )

    .when(
        (F.col("sub_issue") == "Not disclosed as an attempt to collect") &
        (F.col("product") == "Debt collection"),
        "Notification didn't disclose it was an attempt to collect a debt"
    )

    .when(
        (F.col("sub_issue") == "Contacted me after I asked not to") &
        (F.col("product") == "Debt collection"),
        "Contacted you after you asked them to stop"
    )

     .when(
        (F.col("sub_issue") == "Contacted me instead of my attorney") &
        (F.col("product") == "Debt collection"),
        "Contacted you instead of your attorney"
    )

    .when(
        (F.col("sub_issue") == "Talked to a third party about my debt") &
        (F.col("product") == "Debt collection"),
        "Talked to a third-party about your debt"
    )

    .when(
        (F.col("sub_issue") == "Impersonated an attorney or official") &
        (F.col("product") == "Debt collection"),
        "Impersonated attorney, law enforcement, or government official"
    )

    .when(
        (F.col("sub_issue") == "Indicated committed crime not paying") &
        (F.col("product") == "Debt collection"),
        "Indicated you were committing crime by not paying debt"
    )
    .when(
        (F.col("sub_issue") == "Indicated shouldn't respond to lawsuit") &
        (F.col("product") == "Debt collection"),
        "Told you not to respond to a lawsuit they filed against you"
    )

    .when(
        (F.col("sub_issue") == "Seized/Attempted to seize property") &
        (F.col("product") == "Debt collection"),
        "Seized or attempted to seize your property"
    )

    .when(
        (F.col("sub_issue") == "Sued w/o proper notification of suit") &
        (F.col("product") == "Debt collection"),
        "Sued you without properly notifying you of lawsuit"
    )

    .when(
        (F.col("sub_issue") == "Sued where didn't live/sign for debt") &
        (F.col("product") == "Debt collection"),
        "Sued you in a state where you do not live or did not sign for the debt"
    )

    .when(
        (F.col("sub_issue") == "Threatened arrest/jail if do not pay") &
        (F.col("product") == "Debt collection"),
        "Threatened to arrest you or take you to jail if you do not pay"
    )

    .when(
        (F.col("sub_issue") == "Threatened to sue on too old debt") &
        (F.col("product") == "Debt collection"),
        "Threatened to sue you for very old debt"
    )

    .when(
        (F.col("sub_issue") == "Called outside of 8am-9pm") &
        (F.col("product") == "Debt collection"),
        "Called before 8am or after 9pm"
    )

    .when(
        (F.col("sub_issue") == "Used obscene/profane/abusive language") &
        (F.col("product") == "Debt collection"),
        "Used obscene, profane, or other abusive language"
    )

    .when(
        (F.col("sub_issue") == "Can't decrease my monthly payments") &
        (F.col("product") == "Student loan"),
        "Problem lowering your monthly payments"
    )

    .when(
        (F.col("sub_issue") == "Can't get flexible payment options") &
        (F.col("product") == "Student loan"),
        "Can't get other flexible options for repaying your loan"
    )

    .when(
        (F.col("sub_issue") == "Can't temporarily postpone payments") &
        (F.col("product") == "Student loan"),
        "Can't temporarily delay making payments"
    )

    .when(
        (F.col("sub_issue") == "Don't agree with fees charged") &
        (F.col("product") == "Student loan"),
        "Don't agree with the fees charged"
    )

    .when(
        (F.col("sub_issue") == "Having problems with customer service") &
        (F.col("product") == "Student loan"),
        "Problem with customer service"
    )   

    .when(
        (F.col("sub_issue") == "Keep getting calls about my loan") &
        (F.col("product") == "Student loan"),
        "Keep getting calls about your loan"
    )

    .when(
        (F.col("sub_issue") == "Need information about my balance/terms") &
        (F.col("product") == "Student loan"),
        "Need information about your loan balance or loan terms"
    )

    .when(
        (F.col("sub_issue") == "Qualify for a better loan than offered") &
        (F.col("product") == "Student loan"),
        "Qualified for a better loan than the one offered"
    )

    .when(
        (F.col("sub_issue") == "Received bad information about my loan") &
        (F.col("product") == "Student loan"),
        "Received bad information about your loan"
    )

    .when(
        (F.col("sub_issue") == "Trouble with how payments are handled") &
        (F.col("product") == "Student loan"),
        "Trouble with how payments are being handled"
    )
    .otherwise(F.col("sub_issue"))
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 229b7965-ab16-4e9a-b0f8-cdddfaf523ed)

In [30]:
display(
    df_silver_valid
    .filter(
        F.col("sub_issue") != F.col("sub_issue_standardized")
    )
    .groupBy(
        "product",
        "sub_issue",
        "sub_issue_standardized"
    )
    .count()
    .orderBy(
        "product",
        "sub_issue"
    )
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2ad83693-1574-4f64-990e-27566ebe258f)

**COMPANY**

In [31]:
display(
    df_silver_valid
    .groupBy("Company")
    .count()
    .orderBy(F.col("Company").desc())
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 409b1db2-8c53-4cbc-9cd5-9992b428928c)

In [ ]:
# [NBSP] will show nonbreaking spaces.
# [SPACE] will show normal spaces, including repeated spaces.
# [U+00C2] will show an extra encoding character, if present.
# [CURLY] will show curly apostrophes.


display(
    df_silver_valid
    .filter(
        F.col("company").contains("\u00A0") |
        F.col("company").contains("\u00C2") |
        F.col("company").contains("\u2019") |
        F.col("company").rlike(r"\s{2,}") |
        (F.col("company") != F.trim(F.col("company")))
    )
    .select(
        "company",
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.col("company"),
                        "\u00A0",
                        "[NBSP]"
                    ),
                    " ",
                    "[SPACE]"
                ),
                "\u00C2",
                "[U+00C2]"
            ),
            "\u2019",
            "[CURLY]"
        ).alias("characters_made_visible")
    )
    .distinct()
    .orderBy("company")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10a9645d-2007-4a22-8396-92da52339053)

In [ ]:
df_silver_valid = df_silver_valid.withColumnRenamed("Company", "company")


df_silver_valid = df_silver_valid.withColumn(
    "company_clean",
    F.regexp_replace(F.col("company"), "\u00A0", " ")
)


df_silver_valid = df_silver_valid.withColumn(
    "company_clean",
    F.regexp_replace(F.col("company_clean"), r"\s+", " ")
)


df_silver_valid = df_silver_valid.withColumn(
    "company_clean",
    F.trim(F.col("company_clean"))
)


df_silver_valid = df_silver_valid.withColumn(
    "company_clean",
    F.regexp_replace(F.col("company_clean"), "\u2019", "'")
)


display(
    df_silver_valid
    .filter(F.col("company") != F.col("company_clean"))
    .select("company", "company_clean")
    .distinct()
)


StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d7a19db-25c1-4127-8752-33b6643e6ad6)

In [34]:
display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e631ed79-1df8-41e0-8831-f1dc3e3d85b3)

**State & ZIpCODE**

In [35]:
display(
    df_silver_valid
    .groupBy("State")
    .count()
    .orderBy("State")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 57b3011e-8b52-4ec5-b293-e70002297cb1)

| Category | Count | Codes |
|---|---:|---|
| US states | 50 | AL, AK, CA, etc. |
| District of Columbia | 1 | DC |
| US territories | 5 | AS, GU, MP, PR, VI |
| Freely associated countries | 3 | FM, MH, PW |
| Military postal codes | 3 | AA, AE, AP |
| Blank / missing | 1 | Missing State |
| **Total distinct entries** | **63** | |

In [36]:
df_silver_valid = df_silver_valid.withColumnRenamed("State", "state")

df_silver_valid = df_silver_valid.withColumn(
    "state", F.upper(F.trim(F.col("state")))
)

df_silver_valid = df_silver_valid.withColumn(
    "state",
    F.when(F.col("state") == "", F.lit(None)).otherwise(F.col("state"))
)

display(df_silver_valid.groupBy("state").count().orderBy("state"))

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d069b96b-d7b1-4718-97a9-228a61094749)

In [37]:
display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 85d91995-73f8-423b-b249-f74ea7187d4d)

In [38]:
display(df_silver_valid.groupBy("ZIP code").count().orderBy("ZIP code"))

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 238ab033-55f0-4e0b-8e3e-eca76a1bdf02)

| ZIP pattern | Example | Analysis value |
|---|---|---|
| Missing | `NULL` | `NULL` |
| Normal 5-digit | `10001` | `100XX` |
| Normal 5-digit | `00601` | `006XX` |
| Already masked | `100XX` | `100XX` |
| Possible lost leading zero | `1001` | Review first |
| Malformed | `0288-` | `NULL` |

In [39]:
four_digit_zip = (
    df_silver_valid
    .filter(F.trim(F.col("ZIP code").cast("string")).rlike(r"^\d{4}$"))
    .groupBy("state", "ZIP code")
    .count()
    .orderBy("state", "ZIP code")
)
display(four_digit_zip)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 42ec6406-4168-4d76-bb8b-8076d83277aa)

| State | Input | Candidate | Action |
|---|---|---|---|
| MA | 1001 | 01001 | Pad; later mask to 010XX |
| CT | 6001 | 06001 | Pad; later mask to 060XX |
| AE | 9009 | 09009 | Pad; later mask to 090XX |
| NY | 6390 | 06390 | Pad; later mask to 063XX |
| CT | 6390 | 06390 | Leave unresolved; then null |
| CA | 1040 | 01040 | Leave unresolved; then null |

| State / region | Four-digit ZIP examples | What we do | Reason |
|---|---|---|---|
| **AE — military postal region** | `9009`, `9011`, `9102` | Add zero → `09009`, `09011`, `09102` | Candidate prefixes `090–098` match AE. |
| **Puerto Rico (PR)** | `0601`, `0703`, `0901` | Add zero → `00601`, `00703`, `00901` | Candidate prefixes `006`, `007` and `009` match PR. |
| **Virgin Islands (VI)** | `0801`, `0802` | Add zero → `00801`, `00802` | Candidate prefix `008` matches VI. |
| **Massachusetts (MA)** | `1001`, `1002`, `5501` | Add zero → `01001`, `01002`, `05501` | Candidate prefixes `010–027` or `055` match MA. |
| **Rhode Island (RI)** | `2801`, `2903` | Add zero → `02801`, `02903` | Candidate prefixes `028–029` match RI. |
| **New Hampshire (NH)** | `3031`, `3801` | Add zero → `03031`, `03801` | Candidate prefixes `030–038` match NH. |
| **Maine (ME)** | `3901`, `4101` | Add zero → `03901`, `04101` | Candidate prefixes `039–049` match ME. |
| **Vermont (VT)** | `5001`, `5602` | Add zero → `05001`, `05602` | Candidate prefixes `050–054` or `056–059` match VT. |
| **Connecticut (CT) — matching entries** | `6001`, `6002`, `6010` | Add zero → `06001`, `06002`, `06010` | Candidate prefixes `060–069` match CT, except exact ZIP `06390`. |
| **Connecticut (CT) — exception** | `6390` | Leave unresolved; later set ZIP to `NULL` | Candidate `06390` belongs to NY, so our CT rule excludes it. |
| **Connecticut (CT) — other nonmatching entries** | `1590`, `2458`, `2660` | Leave unresolved; later set ZIP to `NULL` | Candidate prefixes do not match CT. |
| **New York (NY) — matching prefix** | `0501` | Add zero → `00501` | Candidate prefix `005` matches NY. |
| **New York (NY) — exception** | `6390` | Add zero → `06390` | Exact candidate `06390` is explicitly included in the NY rule. |
| **New Jersey (NJ)** | `7001`, `8901` | Add zero → `07001`, `08901` | Candidate prefixes `070–089` match NJ. |
| **California (CA)** | `1040`, `1764`, `2084` | Leave unresolved; later set ZIP to `NULL` | Adding zero does not produce a California prefix. |
| **Colorado (CO)** | `6002`, `6035`, `6051` | Leave unresolved; later set ZIP to `NULL` | Adding zero does not produce a Colorado prefix. |
| **Other regions in your pasted list:** AK, AL, AP, AZ, DC, DE, FL, FM, GA, HI, IL, IN, KY, LA | FL: `1852`; GA: `2026`; AZ: `8522` | Leave unresolved; later set ZIP to `NULL` | These regions do not pass our leading-zero rule. |
| **Any listed region with a nonmatching candidate** | PR: `0801`; AE: `9901` | Leave unresolved; later set ZIP to `NULL` | Being a listed region is not enough—the candidate must also match its allowed prefix. |
| **Missing State / region** | `1001` | Leave unresolved; later set ZIP to `NULL` | We cannot establish a State-prefix match. |


In [40]:
location_rows_before = df_silver_valid.count()
df_silver_valid = df_silver_valid.withColumnRenamed("ZIP code", "zip_code")
df_silver_valid = df_silver_valid.withColumn(
    "zip_code", F.trim(F.col("zip_code").cast("string"))
)
display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6a29f4a2-9959-4174-b650-1ca983ca8133)

In [41]:
zip_candidate = F.lpad(F.col("zip_code"), 5, "0") 

zip_prefix = F.when(
    # We are checking whether the original value contains one to four digits.
    F.col("zip_code").rlike(r"^\d{1,4}$"),

    # We are taking the first three characters of the candidate.
    # For 01001, we will get 010, which we are converting to integer 10 for comparison.
    # The full candidate will remain 01001.
    F.substring(zip_candidate, 1, 3).cast("int")
)

valid_leading_zero = (

    # For Puerto Rico, we will allow prefixes 006, 007 and 009.
    ((F.col("state") == "PR") & zip_prefix.isin(6, 7, 9)) |

    # For the Virgin Islands, we will allow prefix 008.
    ((F.col("state") == "VI") & (zip_prefix == 8)) |

    # We are checking for Massachusetts.
    ((F.col("state") == "MA") &
     # We will allow prefixes 010 through 027, or 055.
     (zip_prefix.between(10, 27) | (zip_prefix == 55))) |

    # For Rhode Island, we will allow prefixes 028 through 029.
    ((F.col("state") == "RI") & zip_prefix.between(28, 29)) |

    # For New Hampshire, we will allow prefixes 030 through 038.
    ((F.col("state") == "NH") & zip_prefix.between(30, 38)) |

    # For Maine, we will allow prefixes 039 through 049.
    ((F.col("state") == "ME") & zip_prefix.between(39, 49)) |

    # We are checking for Vermont.
    ((F.col("state") == "VT") &
     # We will allow prefixes 050 through 054, or 056 through 059.
     (zip_prefix.between(50, 54) | zip_prefix.between(56, 59))) |

    # For Connecticut, we are checking prefixes 060 through 069.
    ((F.col("state") == "CT") & zip_prefix.between(60, 69) &
     # We are excluding 06390; we will handle it under New York.
     (zip_candidate != "06390")) |

    # We are checking for New York.
    ((F.col("state") == "NY") &
     # We will allow prefix 005 or the specific candidate 06390.
     ((zip_prefix == 5) | (zip_candidate == "06390"))) |

    # For New Jersey, we will allow prefixes 070 through 089.
    ((F.col("state") == "NJ") & zip_prefix.between(70, 89)) |

    # For the AE military mailing region, we will allow prefixes 090 through 098.
    ((F.col("state") == "AE") & zip_prefix.between(90, 98))
)

df_silver_valid = df_silver_valid.withColumn(
    # We are updating the existing ZIP column.
    "zip_code",

    # We will use the padded candidate only when both checks pass.
    F.when(
        # We are checking for exactly four digits at this stage.
        F.col("zip_code").rlike(r"^\d{4}$") &

        # We are also checking whether the candidate passes our State-and-prefix rule.
        valid_leading_zero,

        # When both checks pass, we will use the candidate.
        # For example, MA with 1001 will become MA with 01001.
        zip_candidate

    # Otherwise, we will keep the current value.
    # We will handle shorter ZIPs and unresolved values in the following cells.
    ).otherwise(F.col("zip_code"))
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 43, Finished, Available, Finished, False)

In [42]:
display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 30016a8d-7722-4722-bddc-bc5619ff19f9)

In [43]:
display(
    df_silver_valid
    .filter(
        F.col("zip_code").rlike(r"^\d{4}$")
    )
    .groupBy(
        "state",
        "zip_code"
    )
    .count()
    .orderBy(
        "state",
        "zip_code"
    )
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f9e77360-3429-4fa2-8608-57a5dde24e61)

In [44]:
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",
    F.when(
        F.col("zip_code").rlike(r"^\d{4}$"),
        F.lit(None)
    ).otherwise(F.col("zip_code"))
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 28822029-44cb-46c4-88fa-7666322b0eac)

In [45]:
display(
    df_silver_valid
    .filter(
        F.col("zip_code").rlike(r"^\d{1,3}$")
    )
    .groupBy(
        "state",
        "zip_code"
    )
    .count()
    .orderBy(
        "state",
        "zip_code"
    )
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 82e89899-85e7-452b-a73e-ae2d350c233f)


| State | Input | Candidate | Result of prefix check |
|---|---|---|---|
| CA | `11` | `00011` | Fail |
| CA | `167` | `00167` | Fail |
| CA | `602` | `00602` | Fail |
| CA | `802` | `00802` | Fail |
| CA | `9` | `00009` | Fail |
| CO | `270` | `00270` | Fail |
| CT | `511` | `00511` | Fail |
| CT | `660` | `00660` | Fail |
| GA | `165` | `00165` | Fail |
| NJ | `623` | `00623` | Fail |
| NJ | `707` | `00707` | Fail |
| NJ | `728` | `00728` | Fail |
| NJ | `872` | `00872` | Fail |
| NY | `457` | `00457` | Fail |
| PA | `703` | `00703` | Fail |
| PR | `39` | `00039` | Fail |
| PR | `466` | `00466` | Fail |
| PR | `536` | `00536` | Fail |
| PR | `601` | `00601` | Pass |
| PR | `602` | `00602` | Pass |
| PR | `603` | `00603` | Pass |
| PR | `604` | `00604` | Pass |
| PR | `605` | `00605` | Pass |
| PR | `61` | `00061` | Fail |
| PR | `610` | `00610` | Pass |
| PR | `611` | `00611` | Pass |
| PR | `612` | `00612` | Pass |
| PR | `613` | `00613` | Pass |
| PR | `614` | `00614` | Pass |
| PR | `616` | `00616` | Pass |
| PR | `617` | `00617` | Pass |
| PR | `622` | `00622` | Pass |
| PR | `623` | `00623` | Pass |
| PR | `624` | `00624` | Pass |
| PR | `627` | `00627` | Pass |
| PR | `637` | `00637` | Pass |
| PR | `638` | `00638` | Pass |
| PR | `639` | `00639` | Pass |
| PR | `641` | `00641` | Pass |
| PR | `646` | `00646` | Pass |
| PR | `650` | `00650` | Pass |
| PR | `652` | `00652` | Pass |
| PR | `656` | `00656` | Pass |
| PR | `657` | `00657` | Pass |
| PR | `659` | `00659` | Pass |
| PR | `660` | `00660` | Pass |
| PR | `662` | `00662` | Pass |
| PR | `664` | `00664` | Pass |
| PR | `667` | `00667` | Pass |
| PR | `669` | `00669` | Pass |
| PR | `674` | `00674` | Pass |
| PR | `676` | `00676` | Pass |
| PR | `677` | `00677` | Pass |
| PR | `678` | `00678` | Pass |
| PR | `680` | `00680` | Pass |
| PR | `681` | `00681` | Pass |
| PR | `682` | `00682` | Pass |
| PR | `683` | `00683` | Pass |
| PR | `685` | `00685` | Pass |
| PR | `687` | `00687` | Pass |
| PR | `688` | `00688` | Pass |
| PR | `690` | `00690` | Pass |
| PR | `692` | `00692` | Pass |
| PR | `693` | `00693` | Pass |
| PR | `694` | `00694` | Pass |
| PR | `698` | `00698` | Pass |
| PR | `703` | `00703` | Pass |
| PR | `704` | `00704` | Pass |
| PR | `705` | `00705` | Pass |
| PR | `707` | `00707` | Pass |
| PR | `711` | `00711` | Pass |
| PR | `715` | `00715` | Pass |
| PR | `716` | `00716` | Pass |
| PR | `717` | `00717` | Pass |
| PR | `718` | `00718` | Pass |
| PR | `719` | `00719` | Pass |
| PR | `720` | `00720` | Pass |
| PR | `721` | `00721` | Pass |
| PR | `723` | `00723` | Pass |
| PR | `725` | `00725` | Pass |
| PR | `726` | `00726` | Pass |
| PR | `727` | `00727` | Pass |
| PR | `728` | `00728` | Pass |
| PR | `729` | `00729` | Pass |
| PR | `730` | `00730` | Pass |
| PR | `731` | `00731` | Pass |
| PR | `732` | `00732` | Pass |
| PR | `733` | `00733` | Pass |
| PR | `735` | `00735` | Pass |
| PR | `736` | `00736` | Pass |
| PR | `737` | `00737` | Pass |
| PR | `738` | `00738` | Pass |
| PR | `739` | `00739` | Pass |
| PR | `741` | `00741` | Pass |
| PR | `745` | `00745` | Pass |
| PR | `751` | `00751` | Pass |
| PR | `754` | `00754` | Pass |
| PR | `757` | `00757` | Pass |
| PR | `759` | `00759` | Pass |
| PR | `765` | `00765` | Pass |
| PR | `766` | `00766` | Pass |
| PR | `767` | `00767` | Pass |
| PR | `769` | `00769` | Pass |
| PR | `771` | `00771` | Pass |
| PR | `772` | `00772` | Pass |
| PR | `773` | `00773` | Pass |
| PR | `775` | `00775` | Pass |
| PR | `777` | `00777` | Pass |
| PR | `778` | `00778` | Pass |
| PR | `780` | `00780` | Pass |
| PR | `782` | `00782` | Pass |
| PR | `783` | `00783` | Pass |
| PR | `784` | `00784` | Pass |
| PR | `785` | `00785` | Pass |
| PR | `788` | `00788` | Pass |
| PR | `791` | `00791` | Pass |
| PR | `792` | `00792` | Pass |
| PR | `794` | `00794` | Pass |
| PR | `795` | `00795` | Pass |
| PR | `821` | `00821` | Fail |
| PR | `901` | `00901` | Pass |
| PR | `902` | `00902` | Pass |
| PR | `903` | `00903` | Pass |
| PR | `907` | `00907` | Pass |
| PR | `908` | `00908` | Pass |
| PR | `909` | `00909` | Pass |
| PR | `910` | `00910` | Pass |
| PR | `911` | `00911` | Pass |
| PR | `912` | `00912` | Pass |
| PR | `913` | `00913` | Pass |
| PR | `914` | `00914` | Pass |
| PR | `915` | `00915` | Pass |
| PR | `917` | `00917` | Pass |
| PR | `918` | `00918` | Pass |
| PR | `919` | `00919` | Pass |
| PR | `920` | `00920` | Pass |
| PR | `921` | `00921` | Pass |
| PR | `922` | `00922` | Pass |
| PR | `923` | `00923` | Pass |
| PR | `924` | `00924` | Pass |
| PR | `925` | `00925` | Pass |
| PR | `926` | `00926` | Pass |
| PR | `927` | `00927` | Pass |
| PR | `928` | `00928` | Pass |
| PR | `929` | `00929` | Pass |
| PR | `931` | `00931` | Pass |
| PR | `934` | `00934` | Pass |
| PR | `936` | `00936` | Pass |
| PR | `937` | `00937` | Pass |
| PR | `940` | `00940` | Pass |
| PR | `949` | `00949` | Pass |
| PR | `95` | `00095` | Fail |
| PR | `950` | `00950` | Pass |
| PR | `951` | `00951` | Pass |
| PR | `952` | `00952` | Pass |
| PR | `953` | `00953` | Pass |
| PR | `956` | `00956` | Pass |
| PR | `957` | `00957` | Pass |
| PR | `958` | `00958` | Pass |
| PR | `959` | `00959` | Pass |
| PR | `960` | `00960` | Pass |
| PR | `961` | `00961` | Pass |
| PR | `962` | `00962` | Pass |
| PR | `963` | `00963` | Pass |
| PR | `965` | `00965` | Pass |
| PR | `966` | `00966` | Pass |
| PR | `968` | `00968` | Pass |
| PR | `969` | `00969` | Pass |
| PR | `970` | `00970` | Pass |
| PR | `971` | `00971` | Pass |
| PR | `976` | `00976` | Pass |
| PR | `977` | `00977` | Pass |
| PR | `978` | `00978` | Pass |
| PR | `979` | `00979` | Pass |
| PR | `982` | `00982` | Pass |
| PR | `983` | `00983` | Pass |
| PR | `984` | `00984` | Pass |
| PR | `985` | `00985` | Pass |
| PR | `986` | `00986` | Pass |
| PR | `987` | `00987` | Pass |
| PR | `988` | `00988` | Pass |
| PW | `627` | `00627` | Fail |
| PW | `913` | `00913` | Fail |
| RI | `859` | `00859` | Fail |
| VI | `801` | `00801` | Pass |
| VI | `802` | `00802` | Pass |
| VI | `803` | `00803` | Pass |
| VI | `804` | `00804` | Pass |
| VI | `815` | `00815` | Pass |
| VI | `820` | `00820` | Pass |
| VI | `821` | `00821` | Pass |
| VI | `822` | `00822` | Pass |
| VI | `823` | `00823` | Pass |
| VI | `824` | `00824` | Pass |
| VI | `830` | `00830` | Pass |
| VI | `831` | `00831` | Pass |
| VI | `840` | `00840` | Pass |
| VI | `841` | `00841` | Pass |
| VI | `850` | `00850` | Pass |
| VI | `851` | `00851` | Pass |


In [46]:
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",
    F.when(
        F.col("zip_code").rlike(r"^\d{1,3}$") & valid_leading_zero,
        zip_candidate
    ).otherwise(F.col("zip_code"))
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 48, Finished, Available, Finished, False)

In [47]:
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",

    F.when(
        F.col("zip_code").rlike(r"^\d{1,3}$"),
        F.lit(None)
    )
    .otherwise(F.col("zip_code"))
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4e1de152-c0df-4c86-8097-6a031669573c)

In [48]:
display(
    df_silver_valid
    .filter(
        F.col("zip_code").isNotNull() &
        ~F.col("zip_code").rlike(r"^\d{5}$") &
        ~F.col("zip_code").rlike(r"^\d{3}XX$")
    )
    .groupBy("zip_code")
    .count()
    .orderBy("zip_code")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dfd1ca8c-7cca-4f63-811a-4c4678631802)

In [49]:
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",

    F.when(
        F.col("zip_code").isNotNull() &
        ~F.col("zip_code").rlike(r"^\d{5}$") &
        ~F.col("zip_code").rlike(r"^\d{3}XX$"),
        F.lit(None)
    )

    .otherwise(F.col("zip_code"))
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 09837ffd-2a7e-48ac-a8fd-d8263d71e372)

In [50]:
display(
    df_silver_valid.select(
        "state", "zip_code",
        F.when(
            F.col("zip_code").rlike(r"^\d{5}$"),
            F.concat(F.substring(F.col("zip_code"), 1, 3), F.lit("XX"))
        ).otherwise(F.col("zip_code")).alias("zip_reporting_preview")
    ).limit(20)
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d29877de-a129-4e30-bcac-76362f77e67a)

In [51]:
display(
    df_silver_valid
    .filter(
        F.col("zip_code").isNotNull() &
        ~F.col("zip_code").rlike(r"^\d{3}XX$")
    )
    .select("zip_code")
    .distinct()
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 758cabf3-30f2-4d54-aedd-043542d7cba7)

In [53]:
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",
    F.when(
        F.col("zip_code").rlike(r"^\d{5}$"),
        F.concat(
            F.substring(F.col("zip_code"), 1, 3),
            F.lit("XX")
        )
    ).otherwise(F.col("zip_code"))
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 55, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f159455-20b3-4fe4-84dc-dec02c8b79c8)

**TAGS**

In [54]:
display(df_silver_valid.groupBy("Tags").count().orderBy(F.col("Tags").desc()))

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 56, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a0da862a-e763-481a-916d-25bc2f95b2b9)

In [55]:
display(
    df_silver_valid
    .groupBy("Tags")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Tags")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 20447ae5-d200-4ec7-b624-c12e5cc45cf1)

In [56]:
df_silver_valid = (
    df_silver_valid.
    withColumnRenamed("Tags","tags")
    .withColumn("tags",F.trim(F.col("tags")))
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c7a44b5-a8d0-4a55-83b3-932a5611f00c)

**SUBMISSION VIA**

In [57]:
display(
    df_silver_valid
    .groupBy("Submitted via")
    .count()
    .orderBy(F.col("count").desc())
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1ed85b73-1ed8-4359-bdaf-6eec77a79cc5)

In [58]:
display(
    df_silver_valid
    .groupBy("Submitted via")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Submitted via")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91b95a6a-5b71-4a81-aca5-6f1d81babbd3)

In [59]:
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed("Submitted via","submitted_via")
    .withColumn("submitted_via",F.trim(F.col("submitted_via")))
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 62, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e034a92b-c9a7-45e6-809d-43a2873292b2)

**Consumer consent provided?**

In [60]:
display(
    df_silver_valid
    .groupBy("Consumer consent provided?")
    .count()
    .orderBy(F.col("count").desc())
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48de2304-fdf9-4b8d-bc3f-57506fef24ef)

In [61]:
display(
    df_silver_valid
    .groupBy("Consumer consent provided?")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Consumer consent provided?")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 65, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4e9c9598-48cf-4802-81d6-6034687c0968)

In [62]:
display(
    df_silver_valid
    .filter(F.col("date_received") < "2015-03-19")
    .groupBy("Consumer consent provided?")
    .count()
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 66, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 31dc4b92-493e-4337-b68b-f9bc3654207a)

In [63]:
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Consumer consent provided?",
        "consumer_consent_provided"
    )
    .withColumn(
        "consumer_consent_provided",
        F.trim(F.col("consumer_consent_provided"))
    )
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 68, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c0d29106-1197-4631-b01f-b101ebc369b0)

**Company public response**

In [64]:
display(
    df_silver_valid
    .groupBy("Company public response")
    .count()
    .orderBy(F.col("count").desc())
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 69, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a9d21576-8a1f-41de-8137-38baff809b77)

In [65]:
display(
    df_silver_valid
    .groupBy("Company public response")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Company public response")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 70, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ba8b1ecc-c921-49c2-a04e-8aec53998d84)

In [66]:
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Company public response",
        "company_public_response"
    )
    .withColumn(
        "company_public_response",
        F.trim(F.col("company_public_response"))
    )
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 72, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f37177b5-3c04-4094-a3ac-619664bcc500)

**Company response to consumer**

In [67]:
display(
    df_silver_valid
    .groupBy("Company response to consumer")
    .count()
    .orderBy(F.col("count").desc())
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 73, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df0b216c-f091-45d0-9e07-8aa597e7410f)

In [68]:
display(
    df_silver_valid
    .groupBy("Company response to consumer")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Company response to consumer")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 75, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 79a0a0db-3104-4656-ba95-4ea8f5e5a1b8)

In [70]:
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Company response to consumer",
        "company_response_to_consumer"
    )
    .withColumn(
        "company_response_to_consumer",
        F.trim(F.col("company_response_to_consumer"))
    )
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 78, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 974cd82c-f568-4219-821d-864d29054ae4)

**Timely response?**

In [71]:
display(
    df_silver_valid
    .groupBy("Timely response?")
    .count()
    .orderBy(F.col("count").desc())
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 79, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 515faaf1-99e8-48f6-a6f9-81ce55b465a0)

In [72]:
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Timely response?",
        "timely_response"
    )
    .withColumn(
        "timely_response",
        F.trim(F.col("timely_response"))
    )
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 80, Finished, Available, Finished, False)

**Consumer disputed?**

In [73]:
display(
    df_silver_valid
    .groupBy("Consumer disputed?")
    .count()
    .orderBy(F.col("count").desc())
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 81, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 64972349-941c-4cf9-b1fc-96ab8664b8ee)

In [74]:
display(
    df_silver_valid
    .groupBy("Consumer disputed?")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Consumer disputed?")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 82, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 70b61b47-1b4b-48e7-85c6-5d6110f8a926)

In [75]:
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Consumer disputed?",
        "consumer_disputed"
    )
    .withColumn(
        "consumer_disputed",
        F.trim(F.col("consumer_disputed"))
    )
)


StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 84, Finished, Available, Finished, False)

In [77]:
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Issue",
        "issue"
    )
)
display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 87, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4eb94113-ee20-4576-97af-5bb62f7d0463)

In [78]:
display(
    df_silver_valid.select(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("Consumer complaint narrative").isNull(),
                1
            ).otherwise(0)
        ).alias("null_count"),

        F.sum(
            F.when(
                F.col("Consumer complaint narrative").isNotNull(),
                1
            ).otherwise(0)
        ).alias("non_null_count")
    )
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 88, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 202fddca-46a8-4b05-9acf-9bc4c9d58d97)

In [80]:
df_silver_valid = df_silver_valid.drop(
    "Consumer complaint narrative"
)

display(df_silver_valid)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 91, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 77c4bf8a-e6b4-411c-9d83-ccf489febf0a)

In [81]:
display(
    df_silver_valid.select(
        F.count("*").alias("total_rows"),
        F.count("complaint_id").alias("non_null_ids"),
        F.countDistinct("complaint_id").alias("distinct_ids")
    )
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 92, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 76a1d1e7-bbee-4249-a1a3-dd806073f81c)

In [82]:
df_silver_valid = df_silver_valid.select(
    "complaint_id",
    "date_received",
    "date_sent_to_company",

    "product",
    "product_standardized",
    "sub_product",
    "sub_product_standardized",

    "issue",
    "issue_standardized",
    "sub_issue",
    "sub_issue_standardized",

    "company",
    "company_clean",

    "state",
    "zip_code",
    "tags",

    "submitted_via",
    "consumer_consent_provided",

    "company_public_response",
    "company_response_to_consumer",

    "timely_response",
    "consumer_disputed"
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 93, Finished, Available, Finished, False)

In [83]:
final_dq = df_silver_valid.select(
    F.count("*").alias("total_rows"),

    F.countDistinct("complaint_id").alias("distinct_complaint_ids"),

    F.sum(
        F.when(F.col("complaint_id").isNull(), 1).otherwise(0)
    ).alias("null_complaint_ids"),

    F.sum(
        F.when(
            F.col("date_sent_to_company") < F.col("date_received"),
            1
        ).otherwise(0)
    ).alias("invalid_date_sequence"),

    F.sum(
        F.when(
            F.col("zip_code").isNotNull() &
            ~F.col("zip_code").rlike(r"^\d{3}XX$"),
            1
        ).otherwise(0)
    ).alias("invalid_zip_format")
)

display(final_dq)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 94, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ed7a373-1665-42e5-a8c7-fcc8cea1152f)

In [84]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 96, Finished, Available, Finished, False)

DataFrame[]

In [85]:
df_silver_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.consumer_complaints")

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 97, Finished, Available, Finished, False)

In [95]:
df_quarantine_final = df_quarantine.select(
    "complaint_id",
    "date_received",
    "date_sent_to_company",
    F.col("Product").alias("product"),
    F.col("Company").alias("company"),
    F.col("State").alias("state"),
    "dq_status",
    "dq_rule",
    "dq_reason"
)

display(df_quarantine_final)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 111, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fb5219d8-65a7-44be-8c01-05a6db584013)

In [96]:
df_quarantine_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.consumer_complaints_quarantine")

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 113, Finished, Available, Finished, False)

In [98]:
df_export = spark.table("silver.consumer_complaints")

print(f"Rows to export: {df_export.count():,}")

(
    df_export.coalesce(1)
    .write.mode("overwrite")
    .option("header", "true")
    .csv("Files/exports/consumer_complaints_full")
)

StatementMeta(, f538e505-d400-4e26-bc68-5713942a39b9, 115, Finished, Available, Finished, False)

Rows to export: 896,931
